In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common, privacy

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat = cfg["catalog"]
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail and not ok else ""))

one = lambda sql: spark.sql(sql).first()[0]
print(cat, "row filter")

In [0]:
# 1. the filter from privacy.yml is on the gold fact
spec = privacy.load()["row_filters"]["fact_bhg_event"]
rf = {r[0]: r for r in spark.sql(f"""SELECT table_name, filter_name, target_columns
        FROM {cat}.information_schema.row_filters WHERE table_schema = 'gold'""").collect()}
row = rf.get("fact_bhg_event")
check("row filter attached to gold.fact_bhg_event", row is not None and row[1].endswith("rf_fact_bhg_event"), row)
check("row filter works on hospital_key", row is not None and "hospital_key" in str(row[2]), row)

# 2. you are in a clear group here, so you see every hospital
fact_h = spark.table(f"{cat}.gold.fact_bhg_event").filter("hospital_key <> -1").select("hospital_key").distinct().count()
dim_h = spark.table(f"{cat}.gold.dim_hospital").filter("hospital_key <> -1").count()
check("clear group sees the events of every hospital", fact_h == dim_h, f"{fact_h} of {dim_h}")

# 3. probe on a throwaway table: a limited group sees only its hospitals
mine = [g for g in ["lh_developer", "lh_teamlead", "lh_manager", "lh_management"]
        if one(f"SELECT is_account_group_member('{g}')")][0]
probe = f"{cat}.ops.tmp_rf_probe"
make = lambda ids: spark.sql("SELECT xxhash64(id) AS hospital_key, id FROM VALUES " + ", ".join(f"('{i}')" for i in ids) + " AS t(id)")
make(["HSP01", "HSP02", "HSP03"]).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(probe)
seen = lambda: sorted(r[0] for r in spark.table(probe).select("id").collect())

fn = privacy.create_row_filter(spark, cfg, env, "probe", {"column": "hospital_key", "limited": {mine: ["HSP01"]}},
                               prefix="tmp_rf_", clear=["lh_nobody"])
spark.sql(f"ALTER TABLE {probe} SET ROW FILTER {fn} ON (hospital_key)")
check("limited group sees only its hospital", seen() == ["HSP01"], seen())

privacy.create_row_filter(spark, cfg, env, "probe", {"column": "hospital_key", "limited": {"lh_nobody": ["HSP01"]}},
                          prefix="tmp_rf_", clear=["lh_nobody"])
check("group not listed sees every row (function replaced while attached)", seen() == ["HSP01", "HSP02", "HSP03"], seen())

try:
    make(["HSP01", "HSP02", "HSP03", "HSP04"]).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(probe)
    still = one(f"SELECT count(*) FROM {cat}.information_schema.row_filters WHERE table_schema = 'ops' AND table_name = 'tmp_rf_probe'")
    check("full overwrite keeps the row filter (gold rebuild path)", still == 1 and len(seen()) == 4, f"filters {still}, rows {len(seen())}")
except Exception as e:
    check("full overwrite keeps the row filter (gold rebuild path)", False, str(e).split("\n")[0][:160])

spark.sql(f"DROP TABLE IF EXISTS {probe}")
spark.sql(f"DROP FUNCTION IF EXISTS {fn}")
print("INFO limited groups in privacy.yml:", spec["limited"])

print(f"\nt20_row_filter: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t20_row_filter failed")